In [14]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import time
from urllib.parse import urljoin

In [131]:
from Bio import Entrez

def get_biosample_raw_xml(biosample_id):
    Entrez.email = "your_email@example.com"  # Replace with your email
    handle = Entrez.efetch(db="BioSample", id=biosample_id, retmode="xml")
    xml_string = handle.read()
    print(xml_string)  # Show the raw XML
    return xml_string

# Test it
xml = get_biosample_raw_xml("SAMEA110755686")



b'<?xml version="1.0" ?>\n<BioSampleSet></BioSampleSet>'


In [132]:
print(get_ena_biosample_metadata("SAMEA110755686"))


Failed request: 400 - {"message":"Invalid fieldName(s) supplied: sample_name,attributes"}

{}


In [133]:
Entrez.email = "your_email@example.com"
search = Entrez.esearch(db="biosample", term="SAMEA110755686")
record = Entrez.read(search)
print(record)

{'Count': '1', 'RetMax': '1', 'RetStart': '0', 'IdList': ['34119800'], 'TranslationSet': [], 'TranslationStack': [{'Term': 'SAMEA110755686[All Fields]', 'Field': 'All Fields', 'Count': '1', 'Explode': 'N'}, 'GROUP'], 'QueryTranslation': 'SAMEA110755686[All Fields]'}


In [134]:
from Bio import Entrez
import xml.dom.minidom

def fetch_and_print_xml(biosample_id):
    Entrez.email = "your_email@example.com"  # Replace with your email
    handle = Entrez.efetch(db="biosample", id=biosample_id, retmode="xml")
    xml_data = handle.read()

    # Pretty print the XML
    dom = xml.dom.minidom.parseString(xml_data)
    pretty_xml = dom.toprettyxml()
    print(pretty_xml[:5000])  # Print the first 5000 chars only

fetch_and_print_xml("34119800")


<?xml version="1.0" ?>
<BioSampleSet>
	<BioSample access="public" publication_date="2023-04-06T00:00:00.000" last_update="2023-04-15T09:24:37.000" submission_date="2023-04-08T10:00:21.760" id="34119800" accession="SAMEA110755686">
		   
		<Ids>
			     
			<Id db="BioSample" is_primary="1">SAMEA110755686</Id>
			     
			<Id db="SRA">ERS12853296</Id>
			   
		</Ids>
		   
		<Description>
			     
			<Title>9852</Title>
			     
			<Organism taxonomy_id="3702" taxonomy_name="Arabidopsis thaliana">
				       
				<OrganismName>Arabidopsis thaliana</OrganismName>
				     
			</Organism>
			     
			<Comment>
				       
				<Paragraph>IP-Ini-0</Paragraph>
				     
			</Comment>
			   
		</Description>
		   
		<Owner>
			     
			<Name>EBI</Name>
			   
		</Owner>
		   
		<Models>
			     
			<Model>Generic</Model>
			   
		</Models>
		   
		<Package display_name="Generic">Generic.1.0</Package>
		   
		<Attributes>
			     
			<Attribute attribute_name="CSNumber">CS76945</Attribute>
			

In [126]:
get_biosample_metadata('SAMEA110755686')

[]


('', '', '', '')

In [215]:
from Bio import Entrez
from xml.etree import ElementTree

# Required by NCBI – add your email here
Entrez.email = "your_email@example.com"

def get_biosample_metadata(biosample_id):
    from Bio import Entrez
    from xml.etree import ElementTree

    Entrez.email = "your_email@example.com"  # Set your email here

    try:
        handle = Entrez.efetch(db="biosample", id=biosample_id, retmode="xml")
        record = handle.read()

        root = ElementTree.fromstring(record)
        ecotype = ""
        sample_name = ""
        submission_date = ""
        cs_number = ""
        lineage = ""
        gl = ""
        sn = ""
        si = ''
        for item in root.findall(".//BioSample"):
            # sample_name might be under 'title' attribute
            sample_name = item.attrib.get("title", "")
            for attr in item.findall(".//Attributes/Attribute"):
                name = attr.attrib.get("attribute_name", "").lower()
                #print(name)
                value = attr.text

                if "ecotype" in name:
                    ecotype = value
                elif "csnumber" in name or "cs number" in name:
                    cs_number = value
                elif "geo_loc_name" in name and not submission_date:
                    submission_date = value
                elif "subspecific genetic lineage name" in name:
                    lineage = value
                elif "geographic location (country and/or sea)" in name:
                    gl = value
                elif "sample name" in name:
                    sn = value

                elif "submitter id" in name:
                    si = value


        
        return ecotype, sample_name, submission_date, cs_number, lineage, gl, sn, si

    except Exception as e:
        print(f"Error retrieving metadata for {biosample_id}: {e}")
        return "", "", "", "", "", "", "", "" 


def search_ncbi_assemblies(species="Arabidopsis thaliana", technology_terms=None):
    import pandas as pd
    import requests

    if technology_terms is None:
        technology_terms = ["PacBio", "Oxford Nanopore"]

    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    tech_query = " OR ".join([f'"{tech}"[All Fields]' for tech in technology_terms])
    term = f'"{species}"[Organism] AND ({tech_query})'

    search_params = {
        "db": "assembly",
        "term": term,
        "retmode": "json",
        "retmax": 10000
    }

    search_url = f"{base_url}esearch.fcgi"
    response = requests.get(search_url, params=search_params)
    data = response.json()

    if 'esearchresult' not in data or 'idlist' not in data['esearchresult']:
        print("No assemblies found or error in query.")
        return []

    assembly_ids = data['esearchresult']['idlist']
    print(f"Found {len(assembly_ids)} assemblies for {species} sequenced with {', '.join(technology_terms)}")

    if not assembly_ids:
        return []

    summary_url = f"{base_url}esummary.fcgi"
    assemblies = []
    batch_size = 500
    for i in range(0, len(assembly_ids), batch_size):
        batch_ids = assembly_ids[i:i + batch_size]
        summary_params = {
            "db": "assembly",
            "id": ",".join(batch_ids),
            "retmode": "json"
        }
        response = requests.get(summary_url, params=summary_params)
        summary_data = response.json()
        if 'result' in summary_data and 'uids' in summary_data['result']:
            for uid in summary_data['result']['uids']:
                assembly_info = summary_data['result'][uid]
                
                biosample_id = assembly_info.get("biosampleid", "")
                gb_bioprojects = assembly_info.get('gb_bioprojects', '')


                if biosample_id:
                    ecotype, sample_name, submission_date, cs_number, lineage, gl, sn, si  = get_biosample_metadata(biosample_id)
                else:
                    ecotype, sample_name, submission_date, cs_number, lineage,  gl, sn, si = "", "", "", "", "", "", "", "" 

                assemblies.append({
                    'accession': assembly_info.get('assemblyaccession', ''),
                    'name': assembly_info.get('assemblyname', ''),
                    'submitter': assembly_info.get('submitterorganization', ''),
                    'date': assembly_info.get('releasedate', ''),
                    'chromosomes': assembly_info.get('chromosomecount', ''),
                    'contigs': assembly_info.get('contign50', ''),
                    'ftp_path': assembly_info.get('ftppath_assembly_rpt', '').replace('_assembly_report.txt', ''),
                    'biosample': biosample_id,
                    'bioprojectaccn': gb_bioprojects[0]['bioprojectaccn'],
                    'ecotype': ecotype,
                    'sample_name': sample_name,
                    'submission_date_loc': submission_date,
                    'cs_number': cs_number,
                    'lineage': lineage,
                    'gl': gl,
                    'sn': sn,
                    'si': si
                    })

    return assemblies


In [216]:
assemblies = search_ncbi_assemblies()
import pandas as pd
df = pd.DataFrame(assemblies)

Found 275 assemblies for Arabidopsis thaliana sequenced with PacBio, Oxford Nanopore


In [193]:
df

,accession,name,submitter,date,chromosomes,contigs,ftp_path,biosample,bioprojectaccn,ecotype,sample_name,submission_date_loc,cs_number,lineage
0,GCA_036942975.1,ASM3694297v1,PRJEB62038,,,14722447,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/036...,38034183,PRJNA1033522,KZ9,,Kazakhstan,,
1,GCA_036942965.1,ASM3694296v1,PRJEB62038,,,13972119,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/036...,38034181,PRJNA1033522,Ita-0,,Morocco,,
2,GCA_036942575.1,ASM3694257v1,PRJEB62038,,,14006506,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/036...,38034179,PRJNA1033522,Cvi-0,,Portugal,,
3,GCA_036942445.1,ASM3694244v1,PRJEB62038,,,14242192,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/036...,38034172,PRJNA1033522,Bla-1,,Spain,,
4,GCA_036942435.1,ASM3694243v1,PRJEB62038,,,14714264,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/036...,38034177,PRJNA1033522,Col-0,,Poland,,
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270,GCA_900303355.1,ONTmin_IT4,MPI FOR DEVELOPMENTAL BIOLOGY,,,12334794,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/900...,8638054,PRJEB21270,,,,,
271,GCA_001753755.2,Athal_Col0Cvi0F1_phased_diploid_1.0,Pacific Biosciences,,,7960654,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/001...,4539663,PRJNA314706,Col-0 X Cvi-0,,not applicable,,
272,GCA_001753755.1,Athal_Col0Cvi0F1_phased_diploid_1.0,Pacific Biosciences,,,7363020,,4539663,PRJNA314706,Col-0 X Cvi-0,,not applicable,,
273,GCA_001651475.1,Ler Assembly,Center for genomic Regulation,,,1193183,ftp://ftp.ncbi.nlm.nih.gov/genomes/all/GCA/001...,4457953,PRJNA311266,Landsberg erecta,,Germany,,


In [207]:
get_biosample_metadata('33237399')

ena-checklist
ena-first-public
ena-last-update
external id
insdc center alias
insdc center name
insdc first public
insdc last update
insdc status
submitter id
common name
geographic location (country and/or sea)
geographic location (latitude)
geographic location (longitude)
growth facility
isolation and growth condition
number of replicons
plant developmental stage
plant growth medium
plant structure
ploidy
sample health state
sample name
scientific_name
subspecific genetic lineage name
subspecific genetic lineage rank


('', '', '', '', 'Ler-0', 'Poland')

In [218]:
df.to_csv('ecotypes_longreadseavail.csv',index=None)

In [195]:
#df = pd.read_csv('ecotypes_longreadseavail.csv')

In [219]:
def get_biosample_metadata(biosample_id):
    from Bio import Entrez
    from xml.etree import ElementTree

    Entrez.email = "your_email@example.com"  # Set your email here

    try:
        handle = Entrez.efetch(db="biosample", id=biosample_id, retmode="xml")
        record = handle.read()
        root = ElementTree.fromstring(record)

        metadata = {}
        for item in root.findall(".//BioSample"):
            metadata['sample_name'] = item.attrib.get("title", "")
            for attr in item.findall(".//Attributes/Attribute"):
                attr_name = attr.attrib.get("attribute_name", "").strip()
                if attr_name:
                    metadata[attr_name] = attr.text

        return metadata

    except Exception as e:
        print(f"Error retrieving metadata for {biosample_id}: {e}")
        return {}

In [220]:
def search_ncbi_assemblies(species="Arabidopsis thaliana", technology_terms=None):
    import pandas as pd
    import requests

    if technology_terms is None:
        technology_terms = ["PacBio", "Oxford Nanopore"]

    base_url = "https://eutils.ncbi.nlm.nih.gov/entrez/eutils/"

    tech_query = " OR ".join([f'"{tech}"[All Fields]' for tech in technology_terms])
    term = f'"{species}"[Organism] AND ({tech_query})'

    search_params = {
        "db": "assembly",
        "term": term,
        "retmode": "json",
        "retmax": 10000
    }

    search_url = f"{base_url}esearch.fcgi"
    response = requests.get(search_url, params=search_params)
    data = response.json()

    if 'esearchresult' not in data or 'idlist' not in data['esearchresult']:
        print("No assemblies found or error in query.")
        return []

    assembly_ids = data['esearchresult']['idlist']
    print(f"Found {len(assembly_ids)} assemblies for {species} sequenced with {', '.join(technology_terms)}")

    if not assembly_ids:
        return []

    summary_url = f"{base_url}esummary.fcgi"
    assemblies = []
    batch_size = 500
    for i in range(0, len(assembly_ids), batch_size):
        batch_ids = assembly_ids[i:i + batch_size]
        summary_params = {
            "db": "assembly",
            "id": ",".join(batch_ids),
            "retmode": "json"
        }
        response = requests.get(summary_url, params=summary_params)
        summary_data = response.json()
        if 'result' in summary_data and 'uids' in summary_data['result']:
            for uid in summary_data['result']['uids']:
                assembly_info = summary_data['result'][uid]
                
                biosample_id = assembly_info.get("biosampleid", "")
                gb_bioprojects = assembly_info.get('gb_bioprojects', '')


                if biosample_id:
                    biosample_metadata = get_biosample_metadata(biosample_id)
                else:
                    biosample_metadata = {}

                assemblies.append({
                    'accession': assembly_info.get('assemblyaccession', ''),
                    'name': assembly_info.get('assemblyname', ''),
                    'submitter': assembly_info.get('submitterorganization', ''),
                    'date': assembly_info.get('releasedate', ''),
                    'chromosomes': assembly_info.get('chromosomecount', ''),
                    'contigs': assembly_info.get('contign50', ''),
                    'ftp_path': assembly_info.get('ftppath_assembly_rpt', '').replace('_assembly_report.txt', ''),
                    'biosample': biosample_id,
                    'bioprojectaccn': gb_bioprojects[0]['bioprojectaccn'],
                    'biosample_metadata': biosample_metadata
                })

    return assemblies


In [221]:
assemblies = search_ncbi_assemblies()
import pandas as pd
df = pd.DataFrame(assemblies)

Found 275 assemblies for Arabidopsis thaliana sequenced with PacBio, Oxford Nanopore


In [223]:

# Assuming `df` is your DataFrame that contains a 'biosample_metadata' column
metadata_expanded = df['biosample_metadata'].apply(pd.Series)

# Merge the metadata fields with the main DataFrame
df_expanded = pd.concat([df.drop(columns='biosample_metadata'), metadata_expanded], axis=1)


In [225]:
df_expanded.to_csv('df_expanded.csv')

In [227]:
curated_lrs_ncbi = pd.read_csv('curated_lrs_ncbi.csv')

In [232]:
curated_lrs_ncbi.columns = ['ecotype', 'accession', 'name', 'submitter', 'contigs', 'ftp_path',
       'biosample', 'bioprojectaccn',
       'geographic_location',
       'latitude', 'longitude',
       'altitude', 'CSNumber', 'EcotypeID', 'ENA-CHECKLIST',
       'INSDC center name', 'collection date', 'Unnamed: 17', 'collected by']

In [234]:
missing_country = curated_lrs_ncbi[
    curated_lrs_ncbi["geographic_location"].isna()
    & curated_lrs_ncbi["latitude"].notna()
    & curated_lrs_ncbi["longitude"].notna()
]

# Check how many rows need country name filled in
missing_country.shape

(168, 19)

In [236]:
from geopy.geocoders import Nominatim
from geopy.extra.rate_limiter import RateLimiter

# Set up the geocoder with rate limiter to respect usage limits
geolocator = Nominatim(user_agent="country_finder")
reverse = RateLimiter(geolocator.reverse, min_delay_seconds=1)

# Create a function to get country from latitude and longitude
def get_country(lat, lon):
    try:
        location = reverse((lat, lon), language='en')
        if location and "country" in location.raw["address"]:
            return location.raw["address"]["country"]
    except:
        return None

# Convert latitude and longitude columns to float
missing_country["latitude"] = missing_country["latitude"].astype(float)
missing_country["longitude"] = missing_country["longitude"].astype(float)

# Apply the reverse geocoding
missing_country["retrieved_country"] = missing_country.apply(
    lambda row: get_country(row["latitude"], row["longitude"]),
    axis=1
)

# Show a sample of the results
missing_country[["latitude", "longitude", "retrieved_country"]].head()

/tmp/ipykernel_923550/451782258.py:18: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_country["latitude"] = missing_country["latitude"].astype(float)
/tmp/ipykernel_923550/451782258.py:19: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  missing_country["longitude"] = missing_country["longitude"].astype(float)
/tmp/ipykernel_923550/451782258.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the 

,latitude,longitude,retrieved_country
49,34.430000,136.310000,Japan
50,43.852723,1.873536,France
51,43.448838,1.790530,France
52,43.909032,1.901077,France
53,44.196005,2.493157,France


In [ ]:
import requests
from bs4 import BeautifulSoup

def search_abrc(cs_number):
    base_url = "https://abrc.osu.edu/stocks"
    params = {
        "field_catalog_number_value": cs_number
    }
    response = requests.get(base_url, params=params)
    if response.status_code != 200:
        return f"Error: {response.status_code}"
    
    soup = BeautifulSoup(response.text, "html.parser")
    
    # Find results container
    result = soup.find("div", class_="view-content")
    if not result:
        return "No results found"

    # Extract text-based description
    items = result.find_all("div", class_="views-row")
    output = []
    for item in items:
        title = item.find("span", class_="field-content").get_text(strip=True)
        output.append(title)
    
    return output

# Example usage:
print(search_abrc("CS76790"))


In [244]:
import numpy as np

In [241]:
curated_lrs_ncbi = curated_lrs_ncbi.merge(missing_country, how = 'left')

In [252]:
curated_lrs_ncbi.loc[curated_lrs_ncbi['geographic_location'].isna(), 'geographic_location'] = curated_lrs_ncbi[curated_lrs_ncbi['geographic_location'].isna()]['retrieved_country']

In [256]:
curated_lrs_ncbi.to_csv('curated_lrs_ncbi.csv',index=None)